In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset, random_split
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.stats import spearmanr, pearsonr
from tqdm import tqdm
import numpy as np
import nibabel as nib 
import csv
import re
import pandas as pd

from a_DenseNetAging import DenseNet3DMRI

In [3]:
def load_nii_as_tensor(nii_path):
      img = nib.load(nii_path)
      data = img.get_fdata()
      data = np.nan_to_num(data)
      #data = (data - np.mean(data)) / (np.std(data) + 1e-5)
      data = torch.tensor(data, dtype = torch.float32)
      data = data.unsqueeze(0)
      return data

In [5]:
def load_dataset_from_nii(img_path, age_dict):
      X_list, y_list = [], []
      subj = []
      for fname in sorted(os.listdir(img_path)):
           if fname.endswith('.nii'): # or fname.endswith('.nii.gz')
                
				# extract subjuct ID
                match = re.search(r'sub-[^_\.]+', fname)
                if not match:
                    continue
                
                subjid = match.group(0)
                if subjid not in age_dict:
                     continue
                
                tensor = load_nii_as_tensor(os.path.join(img_path, fname))
                X_list.append(tensor)
                y_list.append(torch.tensor([age_dict[subjid]], dtype = torch.float32))
                subj.append(subjid)
                
      X = torch.stack(X_list) 
      y = torch.stack(y_list).squeeze()
      
      return TensorDataset(X, y), subj

In [7]:
def train(model, train_loader, val_loader, device, epochs = 30, lr = 1e-4, fold=1, patience=5, save_path='./model'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr = lr, betas = (0.9, 0.99)) # betas from Nature Aging, 2021
    criterion = nn.L1Loss() # mae loss
    best_mae = float('inf')
    os.makedirs(save_path, exist_ok=True)
    model_path = os.path.join(save_path, f'best_mdl_fold{fold}.pt')
    
	# Early stopping tracker
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in tqdm(train_loader, desc = f"[Fold {fold}] Epoch {epoch+1}"):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch).squeeze(dim=1)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        
		# model validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch).squeeze()
                val_preds.extend(y_pred.cpu().numpy())
                val_targets.extend(y_batch.cpu().numpy())
                
        val_mae = mean_absolute_error(val_targets, val_preds)
        pearson_r, _ = pearsonr(val_targets, val_preds)
        spearman_r, _ = spearmanr(val_targets, val_preds)
        print(f'[Fold {fold}] Epoch {epoch+1}: '
              f'Training Loss = {train_loss: .4f},'
              f'Validate MAE = {val_mae: .4f}, '
              f'Pearson r = {pearson_r:.4f}, '
              f'Spearman r = {spearman_r:.4f}')
              
		# save model
        if val_mae < best_mae:
            best_mae = val_mae
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            print(f'[Fold {fold}] model updated. Best validate MAE = {best_mae: .4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'[Fold {fold}] Early stopping at epoch {epoch+1}')
                break
    return model_path

In [9]:
def test(model, test_loader, device, model_path):
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch).squeeze(dim=1)
            preds.extend(y_pred.cpu().numpy())
            targets.extend(y_batch.cpu().numpy())
            
    mae = mean_absolute_error(targets, preds)
    pearson_r, _ = pearsonr(targets, preds)
    spearman_r, _ = spearmanr(targets, preds)

    return preds, targets, mae, pearson_r, spearman_r

In [11]:
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 4
    k_folds = 5
    img_path = 'D:/_workings/projects/DenseNetAging/0_datasets/GMV_CC/mri/'
    age_path = 'D:/_workings/projects/DenseNetAging/0_datasets/GMV_CC/Demograpy_CC.tsv' 
    save_path = 'D:/_workings/projects/DenseNetAging/results/1_model/'   
    age_dict = {}
    
    with open(age_path, newline = '') as tsvfile:
        reader = csv.DictReader(tsvfile, delimiter='\t')
        for row in reader:
            age_dict[row['participant_id']] = float(row['age'])
            
    dataset, subj = load_dataset_from_nii(img_path, age_dict)
    
	# split dataset
    kf = KFold(n_splits = k_folds, shuffle = True, random_state = 1)
    all_test_mae = []
    all_test_pearsonr = []
    all_test_spearmanr = []
    
    for fold, (train_val_idx, test_idx) in enumerate(kf.split(dataset)):
        test_set = Subset(dataset, test_idx)
        test_subj = [subj[i] for i in test_idx]
        remaining_set = Subset(dataset, train_val_idx)
        
        train_len = int(0.75 * len(remaining_set)) # 60% = 0.75 * 0.8; is there some good ideas?
        val_len = len(remaining_set) - train_len
        train_set, val_set = random_split(remaining_set, [train_len, val_len], 
                                          generator = torch.Generator().manual_seed(1))
        
        train_loader = DataLoader(train_set, batch_size = batch_size, shuffle=True, num_workers=12, pin_memory=True)
        val_loader = DataLoader(val_set, batch_size = batch_size, num_workers=12, pin_memory=True)
        test_loader = DataLoader(test_set, batch_size = batch_size, num_workers=12, pin_memory=True)
        
        model = DenseNet3DMRI()
        model_path = train(model, train_loader, val_loader, device, fold = fold + 1, save_path = save_path)
        preds, targets, mae, pearsonr, spearmanr = test(model, test_loader, device, model_path)
        
        all_test_mae.append(mae)
        all_test_pearsonr.append(pearsonr)
        all_test_spearmanr.append(spearmanr)
        print(f'Fold {fold+1} Testing MAE = {mae: .4f}, Pearson r = {pearsonr: .4f}, Spearman r = {spearmanr: .4f}')
        df = pd.DataFrame({'subjid': test_subj, 'pred': preds, 'target': targets})
        df.to_csv(f'results_fold{fold+1}.csv', index=False)
        
    for i, (mae, p_r, s_r) in enumerate(zip(all_test_mae, all_test_pearsonr, all_test_spearmanr)):
        print(f"Fold {i+1}: MAE = {mae: .4f}, Pearson r = {p_r: .4f}, Spearman r = {s_r: .4f}")
    print(f'average MAE: {np.mean(all_test_mae): .4f} ({np.std(all_test_mae):.4f})')   
    print(f'Average Pearson r: {np.mean(all_test_pearsonr):.4f} ({np.std(all_test_pearsonr):.4f})')
    print(f'Average Spearman r: {np.mean(all_test_spearmanr):.4f} ({np.std(all_test_spearmanr):.4f})')       

In [ ]:
if __name__ == "__main__":
    main()

[Fold 1] Epoch 1:  14%|███████████████████████████                                                                                                                                                                  | 14/98 [04:32<23:18, 16.64s/it]